In [ ]:
import sys
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.svm import SVR
from code.Data_Preprocessing import DataPreprocessing_melt as mf
from code.Data_Processing import TargetVariables as tv
from code.Data_Processing import DataProcessing_run as rf
from joblib import Parallel, delayed
from sklearn.model_selection import LeaveOneOut
from sklearn.neural_network import MLPRegressor

In [ ]:
encoder = LabelEncoder()
MAX_HDRS = 69 
MAX_CDI = 90 
df = pd.read_pickle("/planilhas/MFCC35.pkl")
print(f"Patients: {df['Patient_ID'].nunique()}")

- Note: For calculating the standardized Y values using HDRS/CDI raw score data, refer to the worksheet in the data folder of this project's main DOI. [Use "Patient_ID" as a cross-reference.]

R1 - YA + A (Y = Delta_Y) MFCCs 1st

In [ ]:
df_model_YAA_S1 = df.copy()
df_model_YAA_S1 = df_model_YAA_S1.drop(columns=df_model_YAA_S1.loc[:, 'MFCC_Session2_Segment0_MeanCoefficient1':'MFCC_Session8_Segment699_StdCoefficient13'].columns)
df_model_YAA_S1 = tv.standardized_delta_y(df_model_YAA_S1, MAX_HDRS, MAX_CDI)

R2 - YA + A (Y = Delta_Y) MFCCs 1st and 8th

In [ ]:
df_model_YAA_S2 = df.copy()
df_model_YAA_S2 = df_model_YAA_S2.drop(columns=df_model_YAA_S2.loc[:, 'MFCC_Session2_Segment0_MeanCoefficient1':'MFCC_Session7_Segment699_StdCoefficient13'].columns)
df_model_YAA_S2 = tv.standardized_delta_y(df_model_YAA_S2, MAX_HDRS, MAX_CDI)

R3 - YA + A (Y = Delta_Y) MFCCs All sessions

In [ ]:
df_model_YAA_S3 = df.copy()
df_model_YAA_S3 = tv.standardized_delta_y(df_model_YAA_S3, MAX_HDRS, MAX_CDI)

R5 - YA (Y = Delta_HDRS) MFCCs 1st

In [ ]:
df_model_YA_S1 = df.copy()
df_model_YA_S1 = df_model_YA_S1.drop(columns=df_model_YA_S1.loc[:, 'MFCC_Session2_Segment0_MeanCoefficient1':'MFCC_Session8_Segment699_StdCoefficient13'].columns)
df_model_YA_S1 = tv.standardized_delta_HDRS(df_model_YA_S1, MAX_HDRS)

R6 - YA (Y = Delta_HDRS) MFCCs 1st and 8th

In [ ]:
df_model_YA_S2 = df.copy()
df_model_YA_S2 = df_model_YA_S2.drop(columns=df_model_YA_S2.loc[:, 'MFCC_Session2_Segment0_MeanCoefficient1':'MFCC_Session7_Segment699_StdCoefficient13'].columns)
df_model_YA_S2 = tv.standardized_delta_HDRS(df_model_YA_S2, MAX_HDRS)

R7 - YA (Y = Delta_HDRS) MFCCs All sessions

In [ ]:
df_model_YA_S3 = df.copy()
df_model_YA_S3 = tv.standardized_delta_HDRS(df_model_YA_S3, MAX_HDRS)

R9 - A (Y = Delta_CDI) MFCCs 1st

In [ ]:
df_model_A_S1 = df.copy()
df_model_A_S1 = df_model_A_S1.drop(columns=df_model_A_S1.loc[:, 'MFCC_Session2_Segment0_MeanCoefficient1':'MFCC_Session8_Segment699_StdCoefficient13'].columns)
df_model_A_S1 = tv.standardized_delta_CDI(df_model_A_S1, MAX_CDI)

R10 - A (Y = Delta_CDI) MFCCs 1st and 8th

In [ ]:
df_model_A_S2 = df.copy()
df_model_A_S2 = df_model_A_S2.drop(columns=df_model_A_S2.loc[:, 'MFCC_Session2_Segment0_MeanCoefficient1':'MFCC_Session7_Segment699_StdCoefficient13'].columns)
df_model_A_S2 = tv.standardized_delta_CDI(df_model_A_S2, MAX_CDI)

R11 - A (Y = Delta_CDI) MFCCs All sessions

In [ ]:
df_model_A_S3 = df.copy()
df_model_A_S3 = tv.standardized_delta_CDI(df_model_A_S3, MAX_CDI)

____

Melt the dfs considering only the MFCCS segments as features

In [ ]:
meta_YAA = ['Patient_ID', 'Y_Standardized_Delta_Y']
df_model_YAA_S1 = mf.get_mfccs_per_segment_mean_std(df_model_YAA_S1, meta_YAA)
df_model_YAA_S2 = mf.get_mfccs_per_segment_mean_std(df_model_YAA_S2, meta_YAA)
df_model_YAA_S3 = mf.get_mfccs_per_segment_mean_std(df_model_YAA_S3, meta_YAA)

In [ ]:
meta_YA = ['Patient_ID', 'Y_Standardized_Delta_HDRS']
df_model_YA_S1 = mf.get_mfccs_per_segment_mean_std(df_model_YA_S1, meta_YA)
df_model_YA_S2 = mf.get_mfccs_per_segment_mean_std(df_model_YA_S2, meta_YA)
df_model_YA_S3 = mf.get_mfccs_per_segment_mean_std(df_model_YA_S3, meta_YA)

In [ ]:
meta_A = ['Patient_ID', 'Y_Standardized_Delta_CDI']
df_model_A_S1 = mf.get_mfccs_per_segment_mean_std(df_model_A_S1, meta_A)
df_model_A_S2 = mf.get_mfccs_per_segment_mean_std(df_model_A_S2, meta_A)
df_model_A_S3 = mf.get_mfccs_per_segment_mean_std(df_model_A_S3, meta_A)

___

Running the experiments using LOO-CV patient-independent

In [ ]:
summary_results = []
all_results = []
for idx, (df, target_column) in enumerate(datasets):
    print(f"\nProcessing Dataset {idx+1} - Target: {target_column}")
    unique_patients = df['Patient_ID'].unique()
    results_rf = Parallel(n_jobs=32, backend='loky')(
        delayed(rf.reg_process_leave_one_out)(
            patient, df, 'Patient_ID', target_column,
            RandomForestRegressor(random_state=42), rf.reg_param_grid_rf
        ) for patient in unique_patients
    )
    print("RF Done")
    results_xgb = Parallel(n_jobs=4, backend='loky')(
        delayed(rf.reg_process_leave_one_out)(
            patient, df, 'Patient_ID', target_column,
            xgb.XGBRegressor(n_jobs=1, random_state=42), rf.reg_param_grid_xgb
        ) for patient in unique_patients
    )
    print("XGB Done")
    results_svr = Parallel(n_jobs=32, backend='loky')(
        delayed(rf.reg_process_leave_one_out)(
            patient, df, 'Patient_ID', target_column,
            SVR(), rf.reg_param_grid_svr
        ) for patient in unique_patients
    )
    print("SVR Done")
    results_mlp = Parallel(n_jobs=32, backend='loky')(
        delayed(rf.reg_process_leave_one_out)(
            patient, df, 'Patient_ID', target_column,
            MLPRegressor(random_state=42), rf.reg_param_grid_mlp
        ) for patient in unique_patients
    )
    print("MLP Done")
    current_results = results_rf + results_xgb + results_svr + results_mlp
    all_results.extend(current_results)

    df_current = pd.DataFrame(current_results)
    for model_name in df_current["Model"].unique():
        df_model = df_current[df_current["Model"] == model_name]
        summary_results.append({
            "Dataset_Index": idx + 1,
            "Target_Column": target_column,
            "Model": model_name,
            "RMSE_Mean": df_model["RMSE"].mean(),
            "MSE_Mean": df_model["MSE"].mean(),
            "MAE_Mean": df_model["MAE"].mean(),
        })
df_results = pd.DataFrame(all_results)
df_summary = pd.DataFrame(summary_results)